# 12G — Advanced research synthesis and metric tournament

**Part 1 · Research question (Q7).** Which advanced methods outperform simpler physical models under identical cohorts, baselines and evaluation protocols — and what should the next scientific milestone be?

**Part 2 · Not a new hypothesis test.** (i) An evidence registry built from the comparisons each track persisted in one schema (`synthesis.COMPARISON_COLUMNS`) — numbers are read, never re-derived. (ii) The six declared comparative experiments. (iii) ONE common-cohort tournament: every available representation added to M1 on the SAME held-out athletes for the 12C frozen target (and the secondary speed target) — exploratory, multiplicity-adjusted. (iv) Rule-based next-milestone recommendations (`synthesis.recommendations`; no winner preselected).

**No naive meta-analysis.** Different targets, metrics and units are never pooled into one score; the evidence matrix shows text statuses per category. Missing tracks are shown as `execution_pending` / `not_feasible`, never as negative results.

**Prerequisites.** 12.0 and 12C at least; every other executed track is included automatically.

In [ ]:
RUN_MODE = 'full'
SEED = 20261015
FORCE_REBUILD = False
SAVE_TABLES = True
SAVE_FIGURES = True
NUM_WORKERS = 0
RUN_TOURNAMENT = True
RUN_OT_BLOCK = True          # recompute the 12E distance-to-reference features for the tournament cohort
SIZE_FRACTIONS = (0.5, 0.75)
SIZE_REPS = 10

In [ ]:
from nflbdb.data.config import Phase01Config, configure_runtime

cfg = Phase01Config.load()
configure_runtime(cfg)

import numpy as np
import polars as pl
from IPython.display import Markdown, display

from nflbdb.research.advanced_movement import config as amc
from nflbdb.research.advanced_movement.artifacts import ResearchStore
from nflbdb.research.advanced_movement.m11 import combine_drills as cd
from nflbdb.research.advanced_movement.m11 import config as m11c
from nflbdb.research.advanced_movement.m11 import sources as src
from nflbdb.research.advanced_movement.m12 import advanced_evaluation as ae
from nflbdb.research.advanced_movement.m12 import config as m12c
from nflbdb.research.advanced_movement.m12 import distributional_capacity as dc
from nflbdb.research.advanced_movement.m12 import figures as fg
from nflbdb.research.advanced_movement.m12 import foundation as fd
from nflbdb.research.advanced_movement.m12 import functional_movement as fm
from nflbdb.research.advanced_movement.m12 import measurement_error as me
from nflbdb.research.advanced_movement.m12 import movement_transport as mt
from nflbdb.research.advanced_movement.m12 import multiview_traits as mv
from nflbdb.research.advanced_movement.m12 import reporting as rp
from nflbdb.research.advanced_movement.m12 import shared_movement_encoder as se
from nflbdb.research.advanced_movement.m12 import synthesis as sy

controls = amc.NotebookControls(run_mode=RUN_MODE, force_rebuild=FORCE_REBUILD, save_figures=SAVE_FIGURES, save_tables=SAVE_TABLES,
                                seed=SEED, num_workers=NUM_WORKERS).validate()
store = ResearchStore(cfg, controls)
amc.seed_everything(controls.seed)
C = m12c.M12Config().for_mode(controls)
S = src.M11Sources(cfg, store, m11c.M11Config().for_mode(controls))
D = fd.M12Data(store, C, S)
PROV = m12c.provenance('12G', controls, C)
W = store.write_frame
MAN = fd.check_foundation(store, C)
label = 'SMOKE — wiring check, numbers are not results' if controls.smoke else 'FULL'
display(Markdown(f'**Run mode: `{controls.run_mode}`** ({label}) · config `{C.hash()}`'))

## Part 3 · Evidence registry (G1) and track status

In [ ]:
STATUS, COMPS, EVID = sy.collect(store, rp.manifest)
W(STATUS, 'm12g_track_status')
W(COMPS, 'm12g_all_comparisons')
W(EVID, 'm12g_evidence_registry')
display(STATUS)
with pl.Config(tbl_rows=60, tbl_width_chars=260, fmt_str_lengths=60):
    display(EVID.select('method_id', 'track', 'n_athletes', 'reliability', 'stopwatch_r2', 'cross_domain_stat', 'delta_rmse', 'delta_lo', 'delta_hi', 'training_regime', 'evidence_tier', 'compute_cost'))

## Part 4–5 · The six comparative experiments (G3) and the evidence-status matrix (G2)

Each experiment shows the comparisons its track ran on identical athletes, with its own metric. Rows are never combined across experiments.

In [ ]:
EXP = sy.experiment_table(COMPS, STATUS)
MAT = sy.evidence_matrix(EVID, COMPS)
W(EXP, 'm12g_experiments')
W(MAT, 'm12g_evidence_matrix')
with pl.Config(tbl_rows=-1, tbl_width_chars=260, fmt_str_lengths=80):
    display(EXP.select('experiment', 'title', 'comparison', 'metric', 'estimate', 'lo', 'hi', 'n_athletes', 'training_regime', 'primary', 'verdict'))

## Part 6–7 · Common-cohort metric tournament (exploratory)

Cohort: linked athletes with the 12C frozen target AND every static predictor (official tests, three band accelerations, observed top speed, sprint Vmax) — identical rows for every model. Models (each = M1 + one block; fold-fitted where a block learns anything): T_scalar bands · T_sprint_vmax (simple physics) · T_functional FPCA (12A) · T_rc regression-calibrated top speed (12B) · T_multiview (12D selected rule, if executed) · T_ot distance-to-reference (12E) · T_encoder inductive F4 embeddings (12F, if executed). Holm and Benjamini–Hochberg over the tournament family; the best block gets a permutation null and a cohort-size sensitivity.

In [ ]:
TOUR, TOUR2, PERM, SIZE = pl.DataFrame(), pl.DataFrame(), {}, pl.DataFrame()
if RUN_TOURNAMENT and store.has('m12c_targets'):
    FREEZE = dc.read_freeze(store)
    BINS = FREEZE['shared_bins']
    reg = D.m12('m12_cohort_registry')
    off, athf = D.officials(), D.athlete_features()
    POS = {int(i): p for i, p in zip(off['nfl_id'].to_list(), off['position_group'].to_list())}
    BANDS = ['forty.band_2_4_mean_accel', 'forty.band_4_6_mean_accel', 'forty.band_6_8_mean_accel']
    STATIC = [*BANDS, 'forty.v_max_obs', 'forty.sprint_vmax']
    tg = store.read_frame('m12c_targets', fd.NB['12C']).select('nfl_id', 'nfl.q90_shared', 'nfl.game_top_speed_q75')
    data = (off.filter(pl.col('nfl_id').is_in(reg.filter(pl.col('linked'))['nfl_id'].to_list())).join(cd.wide(athf, STATIC), on='nfl_id', how='left')
            .join(tg, on='nfl_id', how='inner').join(D.m12('m12_fold_manifest').select('nfl_id', 'fold'), on='nfl_id', how='inner'))
    data = data.filter(pl.all_horizontal([pl.col(k).is_not_null() for k in STATIC])).sort('nfl_id')
    MODELS = {'M1': ae.Model(list(ae.M1_COLS)), 'T_scalar': ae.Model([*ae.M1_COLS, *BANDS]), 'T_sprint_vmax': ae.Model([*ae.M1_COLS, 'forty.sprint_vmax'])}
    ids_c, Mc, _ = fm.combine_athlete_matrix(D.m12('m12_combine_curves'), BINS, C)
    MODELS['T_functional'] = ae.Model(list(ae.M1_COLS), factory=lambda a: fm.fpca_block(a, ids_c, Mc, C))
    long = me.attempt_long(D.attempt_features(), ['forty.v_max_obs'])
    stats = me.athlete_stats(long)
    MODELS['T_rc'] = ae.Model(list(ae.M1_COLS), factory=lambda a: me.rc_block(a, stats, long, 'forty.v_max_obs', POS))
    if (store.data_dir / 'm12d_selected_representation.json').is_file():
        sel = store.read_json('m12d_selected_representation', fd.NB['12D'])
        mv_ids, mv_X, _, mv_views = mv.athlete_matrix(athf, mv.view_features(athf['feature'].unique().to_list()))
        MODELS['T_multiview'] = ae.Model(list(ae.M1_COLS), factory=lambda a: mv.factor_block(sel['model'], sel['k'], mv_ids, mv_X, mv_views, POS, a, C, C.seed))
    if RUN_OT_BLOCK:
        GRID = mt.Grid(C, BINS)
        ids_t = data['nfl_id'].to_list()
        h = D.hist_athletes(ids_t)
        cubes = dc.build_cubes(h, POS, C)
        del h
        COMB = mt.combine_hists(D.m12('m12_combine_hist').filter(pl.col('nfl_id').is_in(ids_t)), GRID, C, 'forty', 'accelerating')
        NFLD = mt.nfl_hists(cubes, GRID, 'accelerating')
        MODELS['T_ot'] = ae.Model(list(ae.M1_COLS), factory=lambda a: mt.ot_block(a, COMB, NFLD, POS, GRID, C))
    byf = {}
    for f in sorted(set(data['fold'].to_list())):
        nm = f'm12f_inductive_{se.PRIMARY}_f{f}'
        if store.has(nm):
            byf[int(f)] = store.read_frame(nm, fd.NB['12F']).drop('held_out_fold')
    if byf and len(byf) == len(set(data['fold'].to_list())):
        FOLD = dict(zip(data['nfl_id'].to_list(), data['fold'].to_list()))
        MODELS['T_encoder'] = ae.Model(list(ae.M1_COLS), factory=lambda a: se.embedding_block(a, FOLD, byf))
    print('tournament models:', list(MODELS), '· cohort', data.height, 'athletes')
    tabs = []
    for tgt in ('nfl.q90_shared', 'nfl.game_top_speed_q75'):
        res = ae.oof_ladder(data, tgt, MODELS, C, C.seed)
        t = ae.compare(res, [(m, 'M1') for m in MODELS if m != 'M1'], C, C.seed, target=tgt)
        t = ae.add_adjusted(ae.add_adjusted(t, how='holm'), how='bh')
        tabs.append(t)
        if tgt == 'nfl.q90_shared':
            RES_PRIMARY = res
    TOUR, TOUR2 = tabs
    best = TOUR.sort('delta_rmse', descending=True)['model'][0] if TOUR.height else None
    if best:
        PERM = ae.permutation_blocks(data, 'nfl.q90_shared', MODELS['M1'], MODELS[best], C, C.seed, n_perm=min(C.n_perm, 100))
        SIZE = sy.cohort_size_sensitivity(data, 'nfl.q90_shared', MODELS['M1'], MODELS[best], SIZE_FRACTIONS, SIZE_REPS, C, C.seed).with_columns(pl.lit(best).alias('model'))
W(TOUR, 'm12g_tournament')
W(TOUR2, 'm12g_tournament_secondary')
W(pl.DataFrame([PERM]) if PERM else pl.DataFrame(), 'm12g_tournament_permutation')
W(SIZE, 'm12g_cohort_sensitivity')
with pl.Config(tbl_rows=-1, tbl_width_chars=260):
    for t in (TOUR, TOUR2):
        display(t.select('target', 'model', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'delta_pct', 'p_boot_le0_holm', 'p_boot_le0_bh', 'verdict') if t.height else Markdown('_tournament not run (needs 12C)_'))
    display(pl.DataFrame([PERM]) if PERM else Markdown('_no permutation_'))
    display(SIZE.group_by('fraction').agg(pl.col('delta_rmse').mean(), pl.col('delta_rmse').quantile(0.1).alias('q10'), pl.col('delta_rmse').quantile(0.9).alias('q90')).sort('fraction') if SIZE.height else SIZE)

## Part 8 · Research interpretation (G5) and decision (G6)

In [ ]:
REC = sy.recommendations(STATUS, COMPS, TOUR if TOUR.height else None)
W(REC, 'm12g_recommendations')
Q = {'Q1 functional information (12A)': 1, 'Q2 measurement error (12B)': 2, 'Q3 NFL movement distributions (12C)': 3, 'Q4 latent traits (12D)': 4,
     'Q5 distribution matching (12E)': 5, 'Q6 shared representation (12F)': 6}
ans = []
for q, k in Q.items():
    sub = COMPS.filter((pl.col('experiment') == k) & pl.col('primary')) if COMPS.height else COMPS
    tr_ = sy.EXPERIMENTS[k][0]
    st_ = STATUS.filter(pl.col('track') == tr_)['status']
    ans.append({'question': q, 'track_status': st_[0] if st_.len() else m12c.PENDING, 'primary_comparisons': sub.height,
                'supported': int((sub['verdict'] == 'supported').sum()) if sub.height else 0, 'inconclusive': int((sub['verdict'] == 'inconclusive').sum()) if sub.height else 0,
                'reversed': int((sub['verdict'] == 'reversed').sum()) if sub.height else 0})
ans.append({'question': 'Q7 incremental value (12G tournament)', 'track_status': m12c.EXECUTED if TOUR.height else m12c.PENDING, 'primary_comparisons': TOUR.height,
            'supported': int((TOUR['verdict'] == 'model_better').sum()) if TOUR.height else 0, 'inconclusive': int((TOUR['verdict'] == 'inconclusive').sum()) if TOUR.height else 0,
            'reversed': int((TOUR['verdict'] == 'model_worse').sum()) if TOUR.height else 0})
ANS = pl.DataFrame(ans)
W(ANS, 'm12g_question_answers')
with pl.Config(tbl_rows=-1, tbl_width_chars=260, fmt_str_lengths=160):
    display(ANS)
    display(REC)

## Part 9 · Figures (G4)

In [ ]:
st_grid = STATUS.select('track', pl.lit('status').alias('aspect'), 'status')
store.show(fg.fig_state_grid(st_grid, 'track', 'aspect', 'status', fg.STATUS_COLORS, 'Milestone 12 track status', 'Status recorded by each notebook’s run manifest in this run mode (execution_pending = not yet run).', label='status'),
           'AM12_G_01_track_status', {'status': STATUS}, source='m12_run_*.json')
if MAT.height:
    store.show(fg.fig_state_grid(MAT, 'method_id', 'category', 'status', fg.STATUS_COLORS, 'Evidence-status matrix', 'Per method and evidence category: supported · inconclusive · rejected / reversed · descriptive · not tested. Text labels carry the meaning; categories are never added up.', label='status'),
               'AM12_G_02_evidence_matrix', {'matrix': MAT}, publication=True, source='m12g_evidence_matrix')
pc = COMPS.filter(pl.col('primary')).with_columns((pl.col('track') + ' · ' + pl.col('comparison')).alias('label')) if COMPS.height else COMPS
if pc.height:
    store.show(fg.fig_forest(pc, 'label', 'estimate', 'lo', 'hi', 'Declared primary comparisons of every track', 'Estimate with 95 % interval in each comparison’s own units (see the metric column); rows are not comparable across tracks and are not pooled.', 'estimate (track-specific units)', color_by='track'),
               'AM12_G_03_primary_forest', {'primary': pc}, publication=True, source='m12g_all_comparisons')
if TOUR.height:
    tf = pl.concat([TOUR, TOUR2], how='diagonal_relaxed').with_columns((pl.col('target') + ' · ' + pl.col('model') + ' (n=' + pl.col('n_athletes').cast(pl.String) + ')').alias('label'))
    store.show(fg.fig_forest(tf, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Common-cohort tournament: paired improvement over official tests', 'Δ RMSE = RMSE(M1) − RMSE(M1 + block) on IDENTICAL held-out athletes per target, paired athlete bootstrap 95 % CI; exploratory (Holm / BH in the table).', 'Δ RMSE (target units)', color_by='target'),
               'AM12_G_04_tournament', {'tournament': tf}, publication=True, source='m12g_tournament')
rn = EVID.filter(pl.col('reliability').is_not_null()).with_columns((1 - pl.col('stopwatch_r2').fill_null(np.nan)).alias('novelty'), pl.col('track').alias('position_group'))
if rn.filter(pl.col('novelty').is_not_nan()).height:
    store.show(fg.fig_scatter(rn.filter(pl.col('novelty').is_not_nan()), 'reliability', 'novelty', 'Reliability versus novelty', 'Each method: measurement reliability (track-specific: ICC, split-half or test–retest) against 1 − stopwatch R² (information not already in official tests). Colour = track.',
                              'reliability', '1 − stopwatch R²'), 'AM12_G_05_reliability_vs_novelty', {'evidence': rn}, publication=True, source='m12g_evidence_registry')
if SIZE.height:
    sz = SIZE.group_by('fraction').agg(pl.col('delta_rmse').mean().alias('mean'), pl.col('delta_rmse').quantile(0.1).alias('lo'), pl.col('delta_rmse').quantile(0.9).alias('hi'))
    store.show(fg.fig_lines(sz, 'fraction', 'mean', None, 'Cohort-size sensitivity of the best tournament block', f'Δ RMSE of {SIZE["model"][0]} vs M1 on random athlete subsamples (10th–90th percentile band, {SIZE_REPS} subsamples per fraction).', 'share of the common cohort', 'Δ RMSE', 'lo', 'hi', ref=0.0),
               'AM12_G_06_cohort_size', {'size': SIZE}, source='m12g_cohort_sensitivity')
if EXP.height:
    ex = EXP.filter(pl.col('estimate').is_not_null()).with_columns((pl.col('experiment').cast(pl.String) + ' · ' + pl.col('comparison')).alias('label'))
    if ex.height:
        store.show(fg.fig_forest(ex.head(60), 'label', 'estimate', 'lo', 'hi', 'The six comparative experiments', 'All comparisons of experiments 1–6 as persisted by their tracks (own units per row; ≤ 60 rows shown, the table holds all).', 'estimate', color_by='track'),
                   'AM12_G_07_experiments', {'experiments': ex}, source='m12g_experiments')

## Part 10–12 · Reports, reproducibility record, next milestone

In [ ]:
nl = chr(10)
rp.update_doc(store, 'ADVANCED_METHOD_COMPARISON.md', '### Track status' + nl + nl + rp.md_table(STATUS) + nl + '### Experiments 1–6' + nl + nl
              + rp.md_table(EXP, ['experiment', 'title', 'comparison', 'metric', 'estimate', 'lo', 'hi', 'n_athletes', 'training_regime', 'verdict'], max_rows=80)
              + nl + '### Common-cohort tournament (exploratory)' + nl + nl + rp.md_table(TOUR, ['target', 'model', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'p_boot_le0_holm', 'p_boot_le0_bh', 'verdict'])
              + nl + nl + rp.md_table(TOUR2, ['target', 'model', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict']))
neg = COMPS.filter(pl.col('primary') & pl.col('verdict').is_in(['inconclusive', 'reversed', 'insufficient_athletes_for_interval'])) if COMPS.height else COMPS
rp.update_doc(store, 'NEGATIVE_RESULTS.md', '### Primary comparisons that did not support their hypothesis' + nl + nl + rp.md_table(neg, ['track', 'comparison', 'metric', 'estimate', 'lo', 'hi', 'n_athletes', 'verdict'])
              + nl + '### Tournament blocks without an interval above 0' + nl + nl + rp.md_table(TOUR.filter(pl.col('verdict') != 'model_better') if TOUR.height else TOUR, ['target', 'model', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict'])
              + nl + '### Not run / not feasible' + nl + nl + rp.md_table(STATUS.filter(pl.col('status').is_in([m12c.PENDING, m12c.NOT_FEASIBLE, m12c.INCOMPLETE]))))
rp.update_doc(store, 'NEXT_RESEARCH_DIRECTIONS.md', '### Rule-based recommendations (synthesis.recommendations)' + nl + nl + rp.md_table(REC) + nl + '### Questions Q1–Q7' + nl + nl + rp.md_table(ANS))
rp.update_doc(store, 'RESEARCH_OBJECTIVES.md', '### Answers so far (counts of primary verdicts)' + nl + nl + rp.md_table(ANS))
rp.update_doc(store, 'EXPERIMENT_REGISTRY.md', '### Execution status' + nl + nl + rp.md_table(rp.status_table(store)))
repro = rp.status_table(store)
fz = store.read_json('m12c_target_freeze', fd.NB['12C']) if (store.data_dir / 'm12c_target_freeze.json').is_file() else {}
rp.update_doc(store, 'REPRODUCIBILITY.md', '### Run manifests (this run mode)' + nl + nl + rp.md_table(repro) + nl
              + f"### Signatures{nl}{nl}- foundation config `{MAN.get('config_hash')}` · cohort `{MAN.get('cohort_signature')}` · folds `{MAN.get('fold_signature')}` · grids `{MAN.get('grid_signature')}`{nl}"
              + f"- 12C target freeze `{fz.get('freeze_hash', '—')}` ({fz.get('estimator', '—')}, {fz.get('frozen_at', '—')}){nl}- seed {controls.seed} · run mode {controls.run_mode}{nl}")
status = m12c.EXECUTED if TOUR.height else m12c.INCOMPLETE
facts = {'status': status, 'run_mode': controls.run_mode, 'sample': f"tracks executed: {', '.join(STATUS.filter(~pl.col('status').is_in([m12c.PENDING, m12c.NOT_FEASIBLE]))['track'].to_list()) or 'none'}; tournament cohort {RES_PRIMARY.y.size if TOUR.height else 0} athletes.",
         'evidence': [f"{r['question']}: {r['supported']} supported / {r['inconclusive']} inconclusive / {r['reversed']} reversed of {r['primary_comparisons']} (track {r['track_status']})" for r in ANS.iter_rows(named=True)]
                     + [f"recommendation: {r['direction']} — {r['triggered_by']} ({r['strength']})" for r in REC.iter_rows(named=True)],
         'alternatives': ['Tracks use different targets and cohorts by design; only the tournament compares methods on identical athletes.'],
         'relevance': 'Turns six method families into one evidence-based choice of what to pursue, stop or confirm next.',
         'limitations': ['One draft cohort; every gain is exploratory until an independent class confirms it.', 'Tournament blocks are the declared representations of each track, not tuned for the tournament.'],
         'next': [r['direction'] for r in REC.iter_rows(named=True)]}
md = rp.conclusion_markdown('12G', facts)
display(Markdown(md))
rp.write_conclusion(store, '12G', md)
rp.run_manifest(store, '12G', PROV, ['m12g_track_status', 'm12g_evidence_registry', 'm12g_experiments', 'm12g_tournament', 'm12g_recommendations', 'm12g_question_answers'], status)
print('Send back: track status, the experiments table, the tournament tables (+ permutation, cohort size), the recommendations, and the generated reports/milestone12 documents.')